# Fairness Over Time - Phase 4: Fix

In [1]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data import DATA_DIR, ARTIFACTS_DIR, load_acs, make_race_eth, split_data, save_split, load_split
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor
from src.models import make_lr, make_xgb, make_pipeline_for, fit_mitigated, load_models, MODEL_INFO
from src.fairness import (make_sensitive, group_metrics, fairness_summary, fairness_long,
                          bootstrap_gaps, evaluate_all, bootstrap_all, save_results)

warnings.filterwarnings("ignore", module="fairlearn")

SEED = 42

In [2]:
#load data
data = {}
for year in [2016, 2017]:
    X, y, meta = load_acs(year, "TX")
    y = y.squeeze("columns")
    assert X.isna().sum().sum() == 0
    assert len(X) == len(y) == len(meta)

    data[year] = (X, y, meta)
    print(f"{year}: {len(X):,} rows | positive rate {y.mean():.4f} | y type: {type(y).__name__}")

2016: 131,089 rows | positive rate 0.3415 | y type: Series
2017: 133,880 rows | positive rate 0.3539 | y type: Series


In [3]:
X16, y16, meta16 = data[2016]
X17, y17, meta17 = data[2017]

del X, y, meta

In [4]:
#load 4 models
models = load_models()

models.keys()

dict_keys(['lr', 'lr_mitigated', 'xgb', 'xgb_mitigated'])

### The do-nothing baseline

In [22]:
from src.fairness import evaluate_model, to_fix_rows
from src.data import METRICS_PATH, FIX_PATH

S0 = {"lr": "none", "xgb": "none",
      "lr_mitigated": "none_mitigated", "xgb_mitigated": "none_mitigated"}

fix_rows = []   # every strategy appends here; saved once in 4f
for name, strategy in S0.items():
    ev = evaluate_model(models[name], X17, y17, meta17)    # CHECK arguments
    label = MODEL_INFO[name]["model"]                       # CHECK structure -> "LR" / "XGBoost"
    fix_rows.append(to_fix_rows(ev, strategy, label, fit_seconds=0))

s0 = pd.concat(fix_rows, ignore_index=True)
print(len(s0))   # expect 4 × 93 + 4 = 376
s0.groupby(["strategy", "model"]).size()

376


strategy        model  
none            LR         94
                XGBoost    94
none_mitigated  LR         94
                XGBoost    94
dtype: int64

In [6]:
ref = pd.read_csv(METRICS_PATH).query("year == 2017")

new = (s0[s0["metric"] != "fit_seconds"]
       .assign(mitigated=lambda d: d["strategy"].eq("none_mitigated")))

check = new.merge(ref, on=["model", "mitigated", "attribute", "group", "metric"],
                  how="outer", suffixes=("_new", "_ref"), indicator=True)

print(check["_merge"].value_counts())          # want: both = 372, nothing else
diff = (check["value_new"] - check["value_ref"]).abs()
print("max diff:", diff.max())
check.loc[diff > 1e-9, ["model", "mitigated", "attribute", "group", "metric",
                        "value_new", "value_ref"]]

_merge
both          372
left_only       0
right_only      0
Name: count, dtype: int64
max diff: 0.0


,model,mitigated,attribute,group,metric,value_new,value_ref


### S1: One accuracy-maximizing cutoff

In [7]:
from src.models import GlobalThresholdClassifier

GRID = np.round(np.arange(0.05, 0.95, 0.005), 3)
y16_np = y16.to_numpy().astype(int)
s1_info = {}

for name in ["lr", "xgb"]:
    base = models[name]
    label = MODEL_INFO[name]["model"]

    t0 = time.perf_counter()
    s16 = base.predict_proba(X16)[:, 1]                      # 2016 scores, computed once
    accs = np.array([((s16 >= t) == y16_np).mean() for t in GRID])
    best_t = GRID[accs.argmax()]
    fit_seconds = time.perf_counter() - t0

    acc_05 = ((s16 >= 0.5) == y16_np).mean()                 # check vs Phase 2 2016
    s1_info[label] = {"threshold": best_t, "acc16_at_0.5": acc_05,
                      "acc16_best": accs.max(), "fit_seconds": fit_seconds}

    clf = GlobalThresholdClassifier(base, best_t)
    ev = evaluate_model(clf, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "global_cutoff", label, fit_seconds))

pd.DataFrame(s1_info).T.round(4)

,threshold,acc16_at_0.5,acc16_best,fit_seconds
LR,0.45,0.8127,0.8152,0.3788
XGBoost,0.46,0.8191,0.8199,0.3592


In [8]:
s1 = pd.concat(fix_rows, ignore_index=True).query("strategy == 'global_cutoff'")
s0 = pd.concat(fix_rows, ignore_index=True).query("strategy == 'none'")

# AUC must be identical (same scores)
auc = lambda d: d.query("attribute == 'overall' and metric == 'auc'").set_index("model")["value"]
print("AUC S1 - S0:", (auc(s1) - auc(s0)).abs().max())       # want 0.0

# Headline comparison on 2017
key = lambda d, m: d.query("(attribute == 'overall' and metric == @m) or "
                           "(attribute == 'race_eth_x_sex' and metric == @m)").set_index("model")["value"]
pd.DataFrame({m: key(s1, m) - key(s0, m) for m in ["accuracy", "DPD", "EOD"]}).round(4)

AUC S1 - S0: 0.0


,accuracy,DPD,EOD
model,,,
LR,0.0027,0.0085,-0.0450
XGBoost,0.0022,0.0101,-0.0253


In [9]:
from src.models import GroupThresholdClassifier

s2_fitted = {}
for name in ["lr", "xgb"]:
    label = MODEL_INFO[name]["model"]

    t0 = time.perf_counter()
    clf = GroupThresholdClassifier(models[name], seed=SEED).fit(X16, y16)
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(clf, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "group_cutoffs", label, fit_seconds))
    s2_fitted[label] = clf
    print(f"{label}: fit {fit_seconds:.1f}s")

LR: fit 1.9s
XGBoost: fit 1.5s


In [11]:
def rules_table(clf):
    d = clf.to_.interpolated_thresholder_.interpolation_dict
    rows = []
    for g, v in d.items():
        rows.append({"group": g,
                     "p0": v["p0"], "cutoff0": v["operation0"].threshold,
                     "p1": v["p1"], "cutoff1": v["operation1"].threshold})
    return pd.DataFrame(rows).round(3)

rules_table(s2_fitted["LR"])

,group,p0,cutoff0,p1,cutoff1
0,"1.0,Hispanic",0.936,0.280,0.064,0.268
1,"1.0,NH Asian",0.367,0.671,0.633,0.615
2,"1.0,NH Black",0.995,0.379,0.005,0.331
3,"1.0,NH Other",0.428,0.434,0.572,0.401
4,"1.0,NH White",0.708,0.619,0.292,0.608
5,"2.0,Hispanic",0.554,0.170,0.446,0.159
6,"2.0,NH Asian",0.263,0.513,0.737,0.371
7,"2.0,NH Black",0.739,0.243,0.261,0.230
8,"2.0,NH Other",0.921,0.285,0.079,0.189
9,"2.0,NH White",0.203,0.451,0.797,0.390


In [12]:
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))

def headline(df):
    q = ("(attribute == 'overall' and metric in ['accuracy', 'auc']) or "
         "(attribute == 'race_eth_x_sex' and metric in ['DPD', 'EOD'])")
    return (df.query(q)
              .pivot(index=["model", "strategy"], columns="metric", values="value")
              [["accuracy", "auc", "DPD", "EOD"]].round(4))

headline(fix_df)

metric                  accuracy     auc     DPD     EOD
model   strategy                                        
LR      global_cutoff     0.8110  0.8831  0.5381  0.5327
        group_cutoffs     0.7846  0.8831  0.2268  0.1120
        none              0.8084  0.8831  0.5296  0.5778
        none_mitigated    0.7971  0.8006  0.3096  0.2118
XGBoost global_cutoff     0.8172  0.8926  0.5125  0.4824
        group_cutoffs     0.7909  0.8926  0.2149  0.1027
        none              0.8150  0.8926  0.5024  0.5077
        none_mitigated    0.8017  0.7899  0.2883  0.1841

#### Retrain on 2016, evaluate on 2017

In [13]:
from sklearn.base import clone

s3_fitted = {}
for name in ["lr", "xgb"]:
    label = MODEL_INFO[name]["model"]
    pipe = clone(models[name])                 # same settings, unfitted; frozen model untouched

    t0 = time.perf_counter()
    pipe.fit(X16, y16)                         # preprocessor AND model fitted on 2016 only
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(pipe, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "retrain", label, fit_seconds))
    s3_fitted[label] = pipe                    # kept in memory only, not saved (per plan)
    print(f"{label}: fit {fit_seconds:.1f}s")

LR: fit 10.6s
XGBoost: fit 4.9s


In [14]:
# 1. Frozen model untouched: should still give the S0 accuracy (0.8084)
print("frozen LR 2017 acc:", round((models["lr"].predict(X17) == y17).mean(), 4))

# 2. The new preprocessor really was fitted on 2016 (category lists differ from 2014)
for label, pipe in s3_fitted.items():
    n_new = len(pipe.named_steps["pre"].get_feature_names_out())
    print(f"{label}: {n_new} encoded features on 2016 (2014 had 456)")

# 3. Headline comparison
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
headline(fix_df)

frozen LR 2017 acc: 0.8084
LR: 532 encoded features on 2016 (2014 had 456)
XGBoost: 532 encoded features on 2016 (2014 had 456)


metric                  accuracy     auc     DPD     EOD
model   strategy                                        
LR      global_cutoff     0.8110  0.8831  0.5381  0.5327
        group_cutoffs     0.7846  0.8831  0.2268  0.1120
        none              0.8084  0.8831  0.5296  0.5778
        none_mitigated    0.7971  0.8006  0.3096  0.2118
        retrain           0.8133  0.8849  0.5181  0.5117
XGBoost global_cutoff     0.8172  0.8926  0.5125  0.4824
        group_cutoffs     0.7909  0.8926  0.2149  0.1027
        none              0.8150  0.8926  0.5024  0.5077
        none_mitigated    0.8017  0.7899  0.2883  0.1841
        retrain           0.8197  0.8950  0.5046  0.4579

In [15]:
from sklearn.base import clone
from src.models import fit_mitigated, SENSITIVE_COLS

s4_fitted = {}

def run_s4(name):
    label = MODEL_INFO[name]["model"]
    base_clf = clone(models[name].named_steps["model"])    # unfitted LR / XGBoost, same settings

    t0 = time.perf_counter()
    pipe = fit_mitigated(base_clf, X16, y16, X16[SENSITIVE_COLS])   # CHECK vs notebook 01
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(pipe, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "retrain_mitigate", label, fit_seconds))
    s4_fitted[label] = pipe                                 # in memory only, not saved
    print(f"{label}: fit {fit_seconds/60:.1f} min")

In [16]:
run_s4("lr")

LR: fit 8.6 min


In [17]:
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
headline(fix_df).loc["LR"]

metric,accuracy,auc,DPD,EOD
strategy,,,,
global_cutoff,0.8110,0.8831,0.5381,0.5327
group_cutoffs,0.7846,0.8831,0.2268,0.1120
none,0.8084,0.8831,0.5296,0.5778
none_mitigated,0.7971,0.8006,0.3096,0.2118
retrain,0.8133,0.8849,0.5181,0.5117
retrain_mitigate,0.7990,0.8139,0.3064,0.2602


In [18]:
sub = fix_df.query("model == 'LR' and attribute == 'race_eth_x_sex' "
                   "and metric in ['TPR', 'FPR'] "
                   "and strategy in ['none_mitigated', 'retrain_mitigate']")
tbl = sub.pivot(index="group", columns=["metric", "strategy"], values="value").round(3)
tbl.sort_values(("TPR", "retrain_mitigate"))

metric,TPR,FPR,TPR,FPR
strategy,none_mitigated,none_mitigated,retrain_mitigate,retrain_mitigate
group,,,,
NH Other × Female,0.546,0.102,0.516,0.085
NH Black × Female,0.638,0.120,0.578,0.099
Hispanic × Male,0.518,0.078,0.625,0.129
NH Black × Male,0.603,0.115,0.628,0.125
NH White × Female,0.604,0.119,0.658,0.139
Hispanic × Female,0.577,0.076,0.678,0.113
NH White × Male,0.645,0.133,0.705,0.171
NH Asian × Female,0.661,0.106,0.709,0.119


In [19]:
run_s4("xgb")

XGBoost: fit 15.5 min


In [20]:
print("frozen LR 2017 acc:", round((models["lr"].predict(X17) == y17).mean(), 4))   # 0.8084
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
print(fix_df.groupby(["strategy", "model"]).size())   # 12 combinations, 94 rows each
headline(fix_df)

frozen LR 2017 acc: 0.8084
strategy          model  
global_cutoff     LR         94
                  XGBoost    94
group_cutoffs     LR         94
                  XGBoost    94
none              LR         94
                  XGBoost    94
none_mitigated    LR         94
                  XGBoost    94
retrain           LR         94
                  XGBoost    94
retrain_mitigate  LR         94
                  XGBoost    94
dtype: int64


metric                    accuracy     auc     DPD     EOD
model   strategy                                          
LR      global_cutoff       0.8110  0.8831  0.5381  0.5327
        group_cutoffs       0.7846  0.8831  0.2268  0.1120
        none                0.8084  0.8831  0.5296  0.5778
        none_mitigated      0.7971  0.8006  0.3096  0.2118
        retrain             0.8133  0.8849  0.5181  0.5117
        retrain_mitigate    0.7990  0.8139  0.3064  0.2602
XGBoost global_cutoff       0.8172  0.8926  0.5125  0.4824
        group_cutoffs       0.7909  0.8926  0.2149  0.1027
        none                0.8150  0.8926  0.5024  0.5077
        none_mitigated      0.8017  0.7899  0.2883  0.1841
        retrain             0.8197  0.8950  0.5046  0.4579
        retrain_mitigate    0.8057  0.7986  0.2592  0.1098

In [23]:
from src.fairness import save_results

save_results(fix_df, FIX_PATH, keys=("strategy", "model"))

check = pd.read_csv(FIX_PATH)
print(check.shape)                                    # expect (1128, 6) = 12 × 94
print(check.columns.tolist())                         # strategy, model, attribute, group, metric, value
print(check.duplicated(["strategy", "model", "attribute", "group", "metric"]).sum())   # expect 0

(1128, 6)
['strategy', 'model', 'attribute', 'group', 'metric', 'value']
0


In [24]:
   fix_df.query("metric == 'fit_seconds'").pivot(index="strategy", columns="model", values="value").round(1)

model,LR,XGBoost
strategy,,
global_cutoff,0.4,0.4
group_cutoffs,1.9,1.5
none,0.0,0.0
none_mitigated,0.0,0.0
retrain,10.6,4.9
retrain_mitigate,516.0,930.7


In [25]:
STRATS = ["none_mitigated", "retrain_mitigate"]

sub = fix_df.query("model == 'LR' and attribute == 'race_eth_x_sex' "
                   "and metric in ['TPR', 'FPR', 'Count'] and strategy in @STRATS")

tbl = sub.pivot(index="group", columns=["metric", "strategy"], values="value")
tbl = tbl[[("Count", "none_mitigated"),
           ("TPR", "none_mitigated"), ("TPR", "retrain_mitigate"),
           ("FPR", "none_mitigated"), ("FPR", "retrain_mitigate")]]
tbl = tbl.rename(columns={("Count", "none_mitigated"): ("Count", "2017")})

display(tbl.sort_values(("TPR", "retrain_mitigate")).round(3))

# Which gap defines EOD for each

metric                     Count            TPR                   \
strategy          none_mitigated none_mitigated retrain_mitigate   
group                                                              
NH Other × Female         1239.0          0.546            0.516   
NH Black × Female         6511.0          0.638            0.578   
Hispanic × Male          23248.0          0.518            0.625   
NH Black × Male           5726.0          0.603            0.628   
NH White × Female        32246.0          0.604            0.658   
Hispanic × Female        19235.0          0.577            0.678   
NH White × Male          37500.0          0.645            0.705   
NH Asian × Female         3152.0          0.661            0.709   
NH Other × Male           1430.0          0.656            0.719   
NH Asian × Male           3593.0          0.730            0.777   

metric                       FPR                   
strategy          none_mitigated retrain_mitigate  
group                                              
NH Other × Female          0.102            0.085  
NH Black × Female          0.120            0.099  
Hispanic × Male            0.078            0.129  
NH Black × Male            0.115            0.125  
NH White × Female          0.119            0.139  
Hispanic × Female          0.076            0.113  
NH White × Male            0.133            0.171  
NH Asian × Female          0.106            0.119  
NH Other × Male            0.114            0.150  
NH Asian × Male            0.127            0.156